# 합성 이미지 탐지 프로젝트 Colab 실행 런처

이 노트북은 `forensic_project` 폴더 안의 코드를 직접 보여주는 파일이 아니라, **Google Colab에서 우리가 했던 실행 순서 그대로 돌리는 공유용 실행 코드**입니다.

Drive에 아래 파일 3개를 올려두고 실행하세요.

```text
MyDrive/forensic_uploads/
├─ forensic_project_code_fixed_flat.zip
├─ defacto_15000_probe.zip
└─ coco_real_15000.zip
```

출력 결과는 아래에 저장됩니다.

```text
MyDrive/forensic_project/
├─ data/dataset.csv
├─ checkpoints/
├─ heatmaps/
├─ features/
└─ results/
```

## 0. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 1. 경로 설정

In [ ]:
from pathlib import Path

UPLOAD_DIR = Path('/content/drive/MyDrive/forensic_uploads')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/forensic_project')
LOCAL_PROJECT_DIR = Path('/content/forensic_project')
LOCAL_ZIP_DIR = Path('/content/zips')

CODE_ZIP = UPLOAD_DIR / 'forensic_project_code_fixed_flat.zip'
DEFACTO_ZIP = UPLOAD_DIR / 'defacto_15000_probe.zip'
COCO_ZIP = UPLOAD_DIR / 'coco_real_15000.zip'

for p in [CODE_ZIP, DEFACTO_ZIP, COCO_ZIP]:
    print(p, '존재:', p.exists())
    if not p.exists():
        raise FileNotFoundError(f'필수 파일이 없습니다: {p}')

## 2. GPU 확인

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('런타임 → 런타임 유형 변경 → GPU로 바꾸세요.')

In [ ]:
!nvidia-smi

## 3. 패키지 설치
`step00_setup.py` 안의 `drive.mount()` 오류를 피하려고, 여기서 직접 설치합니다.

In [ ]:
!pip -q install segmentation-models-pytorch albumentations lightgbm shap scikit-image timm opencv-python-headless joblib tqdm

## 4. 코드 zip 풀기 + Drive 프로젝트 폴더에 복사

In [ ]:
!rm -rf /content/forensic_project
!mkdir -p /content/forensic_project
!mkdir -p /content/drive/MyDrive/forensic_project

!unzip -o -q /content/drive/MyDrive/forensic_uploads/forensic_project_code_fixed_flat.zip -d /content/forensic_project

# step02~step05가 /content/drive/MyDrive/forensic_project 에서 utils.py 등을 import하므로 Drive에도 복사
!cp /content/forensic_project/*.py /content/drive/MyDrive/forensic_project/
!cp /content/forensic_project/README.txt /content/drive/MyDrive/forensic_project/ 2>/dev/null || true

!ls -al /content/drive/MyDrive/forensic_project | head

## 5. 데이터 zip을 /content로 복사
Drive에서 바로 unzip하면 `Transport endpoint is not connected` 오류가 자주 나므로, 먼저 `/content/zips`로 복사합니다.

In [ ]:
!rm -rf /content/zips
!mkdir -p /content/zips

!cp /content/drive/MyDrive/forensic_uploads/defacto_15000_probe.zip /content/zips/
!cp /content/drive/MyDrive/forensic_uploads/coco_real_15000.zip /content/zips/

!ls -lh /content/zips

## 6. Step 01 — 데이터 압축 해제 + 최종 dataset.csv 생성

In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step01_data_prep.py --clean \
  --defacto_zip /content/zips/defacto_15000_probe.zip \
  --coco_zip /content/zips/coco_real_15000.zip

## 7. dataset.csv 검증
아래 출력이 정확해야 다음으로 넘어갑니다.

In [ ]:
import os
import pandas as pd

csv_path = '/content/drive/MyDrive/forensic_project/data/dataset.csv'
df = pd.read_csv(csv_path)

print('전체:', len(df))
print(pd.crosstab(df['split'], df['label']))
print('이미지 존재율:', df['image_path'].apply(os.path.exists).mean())
print('마스크 존재율:', df['mask_path'].apply(os.path.exists).mean())

leak = df.groupby('group_id')['split'].nunique()
leak = leak[leak > 1]
print('여러 split에 섞인 group_id 수:', len(leak))

assert len(df) == 30000
assert df['image_path'].apply(os.path.exists).all()
assert df['mask_path'].apply(os.path.exists).all()
assert len(leak) == 0
print('검증 통과')

## 8-A. Stage 1 전체 학습 실행

처음부터 3회 repeated hold-out을 모두 돌릴 때 실행합니다.

시간이 오래 걸립니다. GPU인지 반드시 확인하세요.

In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step02_train_stage1.py

## 8-B. 할당량 때문에 끊겼을 때: 특정 Run만 따로 실행

이미 Run 1/3, Run 2/3이 끝났다면 아래 스크립트를 만든 뒤 `run_id=2`만 실행하면 됩니다.

```text
Run 1/3 → run_id 0 → run0_best.pth
Run 2/3 → run_id 1 → run1_best.pth
Run 3/3 → run_id 2 → run2_best.pth
```

In [ ]:
%%writefile /content/drive/MyDrive/forensic_project/step02_run_one.py
import os
import sys
import json
import argparse
import numpy as np
import pandas as pd
import torch

BASE_DIR = "/content/drive/MyDrive/forensic_project"
sys.path.insert(0, BASE_DIR)

import step02_train_stage1 as s2


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--run_id", type=int, required=True, help="0, 1, 2 중 하나")
    args = parser.parse_args()

    if args.run_id not in [0, 1, 2]:
        raise ValueError("run_id는 0, 1, 2 중 하나여야 합니다.")

    seed = s2.CFG["seeds"][args.run_id]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"device: {device}")
    print(f"Run {args.run_id + 1}/3만 실행합니다. seed={seed}")

    df = pd.read_csv(s2.CSV_PATH)
    train_df = df[df["split"] == "train"].reset_index(drop=True)

    missing_img = (~df["image_path"].map(os.path.exists)).sum()
    missing_msk = (~df["mask_path"].map(os.path.exists)).sum()
    if missing_img or missing_msk:
        raise FileNotFoundError(f"경로 오류: missing images={missing_img}, missing masks={missing_msk}")

    groups = train_df["group_id"].unique()
    rng = np.random.RandomState(seed)
    rng.shuffle(groups)

    split_n = int(len(groups) * (1 - s2.CFG["inner_val_ratio"]))
    inner_train_grps = set(groups[:split_n])
    inner_val_grps = set(groups[split_n:])

    inner_train_df = train_df[train_df["group_id"].isin(inner_train_grps)]
    inner_val_df = train_df[train_df["group_id"].isin(inner_val_grps)]

    best_dice, ckpt_path, epoch_log = s2.train_single_run(
        inner_train_df,
        inner_val_df,
        run_id=args.run_id,
        seed=seed,
        device=device
    )

    os.makedirs(f"{BASE_DIR}/results", exist_ok=True)
    log_path = f"{BASE_DIR}/results/stage1_run{args.run_id}_log.json"

    log = {
        "run": args.run_id,
        "seed": seed,
        "best_val_dice": float(best_dice),
        "ckpt_path": ckpt_path,
        "epoch_log": epoch_log
    }

    with open(log_path, "w") as f:
        json.dump(log, f, indent=2, default=str)

    print(f"Run {args.run_id + 1}/3 완료")
    print(f"best dice: {best_dice}")
    print(f"checkpoint: {ckpt_path}")
    print(f"log: {log_path}")


if __name__ == "__main__":
    main()

### Run 3만 실행할 때

In [ ]:
# Run 3/3만 실행할 때 사용
!python -u /content/drive/MyDrive/forensic_project/step02_run_one.py --run_id 2

## 9. 저장된 checkpoint 중 best 선택 + stage1_training_log.json 생성

In [ ]:
%%writefile /content/drive/MyDrive/forensic_project/step02_finalize_from_ckpts.py
import os
import sys
import json
import glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

BASE_DIR = "/content/drive/MyDrive/forensic_project"
sys.path.insert(0, BASE_DIR)

from utils import ForensicDataset, compute_stage1_metrics
import step02_train_stage1 as s2


@torch.no_grad()
def evaluate_ckpt(ckpt_path, val_df, device):
    model = s2.build_model().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    val_ds = ForensicDataset(val_df, mode="val", target_size=s2.CFG["target_size"])
    val_dl = DataLoader(
        val_ds,
        batch_size=s2.CFG["batch_size"],
        shuffle=False,
        num_workers=s2.CFG["num_workers"]
    )

    metrics = {"iou": [], "dice": [], "precision": [], "recall": [], "fp_ratio": []}

    for images, masks, labels in val_dl:
        images = images.to(device)
        preds = torch.sigmoid(model(images)).cpu().numpy()
        gts = masks.numpy()
        labs = labels.numpy()

        for pred, gt, lab in zip(preds, gts, labs):
            m = compute_stage1_metrics(pred[0], gt[0])
            if int(lab) == 1 and gt[0].sum() > 0:
                metrics["iou"].append(m["iou"])
                metrics["dice"].append(m["dice"])
                metrics["precision"].append(m["precision"])
                metrics["recall"].append(m["recall"])
            elif int(lab) == 0 and m["fp_area_ratio"] is not None:
                metrics["fp_ratio"].append(m["fp_area_ratio"])

    def mean(x):
        return float(np.mean(x)) if len(x) > 0 else 0.0

    return {k: mean(v) for k, v in metrics.items()}


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    df = pd.read_csv(f"{BASE_DIR}/data/dataset.csv")
    val_df = df[df["split"] == "validation"].reset_index(drop=True)

    ckpts = sorted(glob.glob(f"{BASE_DIR}/checkpoints/run*_best.pth"))
    if len(ckpts) == 0:
        raise RuntimeError("checkpoint가 없습니다.")

    print("찾은 checkpoint:")
    for c in ckpts:
        print(c)

    results = []
    for ckpt in ckpts:
        print(f"\n평가 중: {ckpt}")
        val_metrics = evaluate_ckpt(ckpt, val_df, device)
        print(val_metrics)
        results.append({
            "ckpt_path": ckpt,
            "val_metrics": val_metrics,
            "external_val_dice": val_metrics["dice"]
        })

    best = max(results, key=lambda x: x["external_val_dice"])

    log = {
        "note": "Checkpoints were trained separately or selected after repeated hold-out training.",
        "available_checkpoints": results,
        "best_ckpt_path": best["ckpt_path"],
        "best_overall_dice": best["external_val_dice"],
        "mean_dice": float(np.mean([r["external_val_dice"] for r in results])),
        "std_dice": float(np.std([r["external_val_dice"] for r in results])),
        "val_metrics": best["val_metrics"]
    }

    out_path = f"{BASE_DIR}/results/stage1_training_log.json"
    os.makedirs(f"{BASE_DIR}/results", exist_ok=True)

    with open(out_path, "w") as f:
        json.dump(log, f, indent=2, default=str)

    print("\n최종 선택 checkpoint:", best["ckpt_path"])
    print("stage1_training_log 저장:", out_path)


if __name__ == "__main__":
    main()

In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step02_finalize_from_ckpts.py

## 10. Step 03 — Heatmap 생성 + Stage 2 feature 추출

In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step03_heatmap_features.py

## 11. Step 04 — Stage 2 LightGBM 학습

In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step04_train_stage2.py

## 12. Step 05 — 최종 평가 + SHAP + 시각화

In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step05_evaluate.py

## 13. 결과 파일 확인

In [ ]:
!echo '--- checkpoints ---'
!ls -lh /content/drive/MyDrive/forensic_project/checkpoints || true
!echo '--- results ---'
!ls -lh /content/drive/MyDrive/forensic_project/results || true
!echo '--- visualizations ---'
!ls -lh /content/drive/MyDrive/forensic_project/results/visualizations || true

## 14. 결과 zip으로 묶기

In [ ]:
!cd /content/drive/MyDrive && zip -qr /content/forensic_project_results.zip \
  forensic_project/data/dataset.csv \
  forensic_project/checkpoints \
  forensic_project/results \
  forensic_project/features \
  forensic_project/heatmaps

!ls -lh /content/forensic_project_results.zip

## 선택 사항: v2 Stage 1 파일을 따로 받은 경우
`step02_train_stage1_v2.py`는 U-Net++/512용이라 기존 step03도 같이 수정해야 합니다. 과제 완주 우선이면 기본 v1 전체 파이프라인을 먼저 끝낸 뒤 v2를 시도하세요.